In [2]:
import warnings
import pandas as pd
from pcntoolkit import (
    NormativeModel,
    NormData,
    Runner,
    dataio,
)
import pcntoolkit.util.output
from pcntoolkit.dataio.fileio import save as ptksave
import logging
import os
import dill #necessary to pickle norm_data objects
import pickle
import sys
import nibabel as nib
from nibabel import processing
import joblib
from joblib import Parallel, delayed
import numpy as np 
from contextlib import redirect_stdout, redirect_stderr
import requests
import xml.etree.ElementTree as ET
from urllib.parse import unquote
import zipfile

# Suppress some annoying warnings and logs
pymc_logger = logging.getLogger("pymc")
pymc_logger.setLevel(logging.WARNING)
pymc_logger.propagate = False
warnings.simplefilter(action="ignore", category=FutureWarning)
pd.options.mode.chained_assignment = None  # default='warn'
pcntoolkit.util.output.Output.set_show_messages(False)

# Download and unzip the models 

First, we extract the voxelwise models from the public repository.

In [32]:
# Create and set the working directory where all ZIP files will be downloaded
root_dir = '/path/to/root/dir/'
wdir = os.path.join(root_dir, "voxelwise_charts")
os.makedirs(wdir, exist_ok=True)
os.chdir(wdir)

# Define the WebDAV URL and authentication credentials for Surfdrive
BASE_URL = "https://surfdrive.surf.nl/public.php/webdav/"
SHARE_TOKEN = "Mb6mZyFmJeCaPcZ"
PASSWORD = ""

# Send a PROPFIND request to list the contents of the remote directory
headers = {"Depth": "1"}
r = requests.request("PROPFIND", BASE_URL, auth=(SHARE_TOKEN, PASSWORD), headers=headers)

# Parse the XML response returned by the WebDAV server
tree = ET.fromstring(r.content)

# Extract the names of all voxelwise model folders in the remote directory
folders = []
for elem in tree.iter():
    if elem.tag.endswith("href"):
        path = unquote(elem.text.rstrip("/"))
        name = path.split("/")[-1]
        if "vox" in name.lower(): #get the voxelwise models
            folders.append(name)

#send another PROPFIND request to get the zip filenames inside of the model folders, then download each
for folder in folders:
    zip_files = []
    BASE_URL = "https://surfdrive.surf.nl/public.php/webdav/" + folder + '/'
    r = requests.request("PROPFIND", BASE_URL, auth=(SHARE_TOKEN, PASSWORD), headers=headers)
    tree = ET.fromstring(r.content)
    for elem in tree.iter():
        if elem.tag.endswith("href"):
            name = unquote(elem.text.split("/")[-1])
            if name.lower().endswith(".zip"):
                zip_files.append(name)

    for fname in zip_files:
        url = BASE_URL + fname
        resp = requests.get(url, auth=(SHARE_TOKEN, PASSWORD), stream=True)
    
        # Skip files that could not be downloaded successfully
        if resp.status_code != 200:
            continue
    
        out_path = os.path.join(wdir, fname)
        with open(out_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

['BLRw_gmv_vox_lifespan_53K_50sites', 'BLRw_jd_vox_lifespan_58K_52sites', 'BLRw_wmv_vox_lifespan_53K_50sites']
['BLR_aff_nonlin_mod_gmv_labelmask_2mm_thresh0.5_smoothed_fwhm6_models.zip']
['BLR_aff_nonlin_log_jacs_wholebrain_2mm_models.zip']
['BLR_aff_nonlin_mod_wmv_2mm_thresh0.5_smoothed_fwhm6_models.zip']


Then, we unzip our chosen voxelwise model (~30-70GB of storage space needed).

In [35]:
# Define paths to the zipped model file and the directory where it will be extracted
zip_path = os.path.join(wdir, "BLR_aff_nonlin_mod_wmv_2mm_thresh0.5_smoothed_fwhm6_models.zip")
model_name = "BLRw_wmv_vox_lifespan_53K_50sites"#simpler model name
proc_dir = os.path.join(wdir, "models", model_name) 

# Unzip the pretrained model if it has not already been extracted
if not os.path.exists(model_dir):
    os.makedirs(proc_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(proc_dir)


# Global settings 

#### Some names and directories

N.B.: Transfer in BLR is implemented as a post-hoc adjustment of predicted normative values on the new dataset with an offset correction estimated for new batch effect values, for example new sites. The parameters of the trained model remain unchanged.

Extend in BLR is implemented as a full re-estimation of the model, using synthetized data approximating the initial training data, merged with the new dataset. It is much more computationally expensive. The rest of the guidelines in this notebook will assume that transfer is used.

In [ ]:
modality = 'wmv'#choose brain measure ('jd', 'gmv', 'wmv')
function = 'transfer' #transfer or extend.

model_dir = os.path.join(proc_dir,'models') #where the batches of trained models live
out_dir = os.path.join(proc_dir,function) # where the output will go
venv_path = os.path.join(os.path.dirname(os.path.dirname(sys.executable))) #path to activate venv with pcntoolkit - needs to be version 1.1.2 or above
tpl_dir = "/path/to/tpl/dir" #where the masks live

if not (os.path.exists(os.path.join(out_dir, 'logs'))):
    os.makedirs(os.path.join(out_dir, 'logs'))

#### Settings from the estimated models - must not be changed

Transfer of the voxelwise models require the voxelwise masks derived from the MNI152NLin2009cSym template resampled to 2mm, available on the github repository in /template_data/.

In [ ]:
#needed to extract brain data from niftis
if modality == 'jd':# binary-masked 2mm templates
    mask_nii = os.path.join(tpl_dir, 'tpl-MNI152NLin2009cSym_res-2_T1w_mask_BrainExtractionBrain.nii.gz')
    smooth = False
    file_name = 'T1_BrainNorm_Jacobian.nii.gz'
if modality == 'gmv':
    mask_nii = os.path.join(tpl_dir,'tpl-MNI152NLin2009cSym_res-2_label-GM_mask_probseg.nii.gz')
    smooth = True
    file_name = 'T1_BrainNorm_GMV.nii.gz'
if modality == 'wmv':
    mask_nii = os.path.join(tpl_dir,'tpl-MNI152NLin2009cSym_res-2_label-WM_mask_probseg.nii.gz')
    smooth = True
    file_name = 'T1_BrainNorm_WMV.nii.gz'

#some details from the already estimated model
ex_nii = mask_nii #used to visualize outputs
mask = dataio.fileio.load(mask_nii, mask=mask_nii, vol=False).T #load all nonzero voxels of the mask
n_voxels = len(mask) #total number of voxels
voxel_indices = list(range(n_voxels))
prev_batch_size = 150 #how many voxels per batch
prev_n_batches = n_voxels // prev_batch_size + int(n_voxels % prev_batch_size != 0) #how many batches

#### New settings that can be adjusted

Voxelwise models were estimated using batches of 150 voxels, as it resulted in a comfortable  balance of 1 core-4GB of RAM per batch for model estimation in the Donders cluster.
Here, the batch size for the transfer can be changed depending on available computational resources (local config / cluster specs etc.).

We also specify how to split the transfer dataset into calibration vs. test sets - a reasonable calibration sample size must be used to adjust for the batch effects.

In [ ]:
new_batch_size = 75 #(adjust as needed)
new_n_batches = n_voxels // new_batch_size + int(n_voxels %new_batch_size != 0) 
split=[0.1, 0.9] #a random seed is already set inside the splitting utility of the toolkit, so the split will be stable

# Data preparation: grab demographic data from the new dataset 

In [ ]:
data_dir = '/path/to/data/folder/
df_dem = pd.read_csv(os.path.join(data_dir,'/path/to/demographics/data.csv'))

#some cleanup is probably needed 
df_dem.rename(columns={'Age': 'age','participant' : 'sub_id'}, inplace=True) #adjust depending on actual column names
df_dem['sex'] = df_dem['sex'].astype(int).map({0: 'F', 1: 'M'})# required format is males as M and females as F to align with trained model - although in v1.1.2 values are still considered unobserved and adjusted for, this might change in future implementations
df_dem['site'] = 'name_your_site' #optional - rename as str rather than int or float to avoid potential downstream problems

df_dem = df_dem [['sub_id','age','sex','site']]
df_dem = df_dem .dropna() #remove rows that have Nan for anything
df_dem.set_index('sub_id',inplace=True)

#these will go to the models
covariates = ["age"]
batch_effects = ["sex", "site"]

#get excluded subjects out
excluded = pd.read_csv(os.path.join(data_dir,'/path/to/list/of/excluded/subjects/IDs.txt'), header = None)
have_scans = pd.read_csv(os.path.join(data_dir,'/path/to/list/of/all/subjects/IDS/with/available/nifti.txt'), header = None)[0] #grab these with a glob.glob search
excluded = excluded.astype(str)
have_scans = have_scans.astype(str).to_list()
excluded_no_scans = pd.DataFrame(df_dem[~df_dem.index.isin(have_scans)].index) #exclude people that have covariates but no brain data
excluded_no_scans.columns=[0] #make it concatenable with the other one 
excluded = pd.concat([excluded, excluded_no_scans],ignore_index = True)
excluded = excluded.values.flatten().tolist()
df_dem = df_dem.drop(excluded, errors='ignore')

#grab image paths for all included subjects
f_pattern =os.path.join(data_dir, 'subjects','{}','T1_ants_MNI152NLin2009cSym', file_name) #adjust as needed
sub_paths = [f_pattern.format(sub) for sub in sorted(df_dem.index.tolist())]

# Utility : prepare extraction of voxel values from nifti images

Dynamically resample to 2mm without saving the new scan to save diskspace, optionally smooth, and flatten the nifti image into an array to extract voxel values within our mask.

In [ ]:
def load_file(sub_path, mask_nii, smooth):
    img = nib.load(sub_path)
    img = processing.resample_to_output(img, [2, 2, 2]) #2mmm resampling
    if smooth:#this will be used for GMV models
        img = nil.processing.smooth_image(img,6) #fwhm 6mmm
    data = img.get_fdata()
    data = dataio.fileio.vol2vec(data, nib.load(mask_nii).get_fdata())
    return data

# Warning for all parallelized steps 
##### There are chances of joblib.Parallel not running/being very slow on jupyter because of the system overhead and job dispatch. 

If that happens, we can try using the ipyparallel module instead, but the fastest option will be to copy the necessary code from this notebook and paste in a standalone python script that can then be run in a cell below with !pythonscript_name.py (or directly in terminal). 

# Data prep: grab the nifti data, option 1 - if we have a small (N<500) dataset or a large RAM
Just load all niftis at once and then build the norm_data objects.

In [ ]:
#load scans of all included subjects
resp_data = Parallel(n_jobs=-1,verbose =5)(delayed(load_file)(sub_path, mask_nii, smooth)for sub_path in sub_paths
                                           
#build norm_data objects from batches of voxels and save them
for b in range(new_n_batches):
    batch = f'batch_{b}'
    print(f"Building norm object for {batch}")
    if not os.path.exists(os.path.join(out_dir,batch)):
        os.makedirs((os.path.join(out_dir,batch)))
    
    start = b * new_batch_size
    end = min(start + new_batch_size, n_voxels)
    batch_resp_data = [array[start:end] for array in resp_data]
        
    norm_data_path = os.path.join(out_dir, batch, 'norm_data.pkl')
    norm_data = NormData.from_ndarrays(
        name=str(len(df_dem)) + "_subjects", #must not have spaces otherwise runner job submission will fail
        X = np.array(df_dem.loc[:, covariates]).squeeze(), 
        Y = np.vstack(batch_resp_data), 
        batch_effects=np.array(df_dem.loc[:, batch_effects]), 
        subject_ids = np.array(df_dem.index)
        )

    #rename variables to align with trained model
    norm_data = norm_data.assign_coords(covariates=covariates)
    norm_data = norm_data.assign_coords(batch_effect_dims=batch_effects)
    norm_data.register_batch_effects() #updates the cached stuff dependent on batch effects names
    norm_data = norm_data.assign_coords(response_vars=[f'voxel_{x}' for x in voxel_indices[start:start+vox_batch_size]])
  
    with open(norm_data_path, 'wb') as f:
        dill.dump(norm_data,f)

# Data prep: grab the nifti data, option 2 - if we're running this from a local machine with limited specs and/or we have a large dataset.
To avoid overloading the RAM with all the scans, we need to load the nifti data in chunks, and populate the batches of voxels iteratively, before building the norm_data objects.

In [ ]:
# Initialize directories and empty files to store batch data
for b in range(new_n_batches):
    batch = 'batch_' + str(b)
    if not os.path.exists(os.path.join(out_dir,batch):
        os.makedirs(os.path.join(out_dir,batch))
        os.makedirs(os.path.join(out_dir,batch,'subjects_chunks')

chunk_size = 500 #how many niftis scans will be loaded at once - adjust based on RAM limitations

#load niftis from chunks of subjects
for chunk_start in range(0, len(sub_paths), chunk_size):
    chunk_paths = sub_paths[chunk_start:chunk_start+chunk_size] 
    current_chunk_size = len(chunk_paths)#this accounts for the last chunk, which will have a smaller number of subjects
    print(f"Loading chunk: subjects {chunk_start} to {chunk_start + current_chunk_size}")

    chunk_data = np.zeros((current_chunk_size, n_voxels), dtype=np.float64)
    data = Parallel(n_jobs=-1,verbose =5)(delayed(load_file)(subpath, mask_nii, smooth) for subpath in chunk_paths) #long step here
    chunk_data = np.stack(data)
    del data #saves some RAM

    #slice each chunk into voxel batches and save those
    print("Saving chunk data to voxel batches...")
    for b in range(new_n_batches):
        batch = f'batch_{b}'
        start = b * new_batch_size
        end = min(start + new_batch_size, n_voxels)
        batch_data = chunk_data[:, start:end]
        
        chunk_name = str(chunk_start)+ '_'+str(chunk_start + current_chunk_size)
        data_path = os.path.join(out_dir,batch,'subjects_chunks', f'subjects_{chunk_name}.pkl')
        with open(data_path, 'wb') as f:
            joblib.dump(batch_data, f)

# now concatenate chunks inside each batch to get the full voxel data per batch and then build the norm_data with it
for b in range(new_n_batches):
    full_batch_data = []
    batch = 'batch_' + str(b)
    print(f"Concatenating chunk data in {batch}...")
    
    #concatenate all chunks in the batch
    for chunk_start in range(0, len(sub_paths), chunk_size):
        chunk_paths = sub_paths[chunk_start:chunk_start+chunk_size]
        current_chunk_size = len(chunk_paths)
        chunk_name = str(chunk_start)+ '_'+str(chunk_start + current_chunk_size)
        data_path = os.path.join(out_dir,batch,'subjects_chunks', f'subjects_{chunk_name}.pkl')
        with open(data_path, 'rb') as f:
            chunk_data = joblib.load(f)
            full_batch_data.extend(chunk_data)
            
    #build the norm_data
    print(f"Building norm object for {batch}")
    norm_data_path = os.path.join(out_dir, batch, 'norm_data.pkl')

    norm_data = NormData.from_ndarrays(
        name=str(len(df_dem)) + "_subjects", #must not have spaces other job submission might fail
        X = np.array(df_dem.loc[:, covariates]).squeeze(), 
        Y = np.vstack(full_batch_data), 
        batch_effects=np.array(df_dem.loc[:, batch_effects]), 
        subject_ids = np.array(df_dem.index)
        )

    #rename variables to align with trained model
    norm_data = norm_data.assign_coords(covariates=covariates)
    norm_data = norm_data.assign_coords(batch_effect_dims=batch_effects)
    norm_data.register_batch_effects() #updates the cached stuff dependent on batch effects names
    norm_data = norm_data.assign_coords(response_vars=[f'voxel_{x}' for x in voxel_indices[start:start+vox_batch_size]])

    with open(norm_data_path, 'wb') as f:
        dill.dump(norm_data,f)
    
###optional cleanup : remove saved chunks to save diskspace - do it when we're sure the batch/norm_data building proceeded correctly otherwise you'll have to load all niftis again
# import shutil
# for b in range(new_n_batches):
#     batch = 'batch_' + str(b)
#     if os.path.exists(os.path.join(out_dir,batch,'subjects_chunks')):
#         shutil.rmtree(os.path.join(out_dir,batch,'subjects_chunks'))


# Data prep: optional step for mixed normative/clinical data 
If there are clinical subjects we're interested in, they should be put in the test data.

In [ ]:
#optional - if there are clinical subjects we're interested in, they should be explicitly put in the test data so we grab them here
clinical_set =True
clinical=pd.read_csv(os.path.join(data_dir, 'path/to/clinical/subject/list/of/IDs.txt'), header = None)
clin_subs = clinical.iloc[:,0].values


def put_patients_in_test_set(clin_subs, norm_data, split):
    #separate the clinical subjects from norm_data 
    clin_subs_existing = [s for s in clin_subs if s in norm_data.subject_ids.values]
    clin_subset = norm_data.where(norm_data.subject_ids.isin(clin_subs_existing),drop = True)
    remaining = norm_data.where(~norm_data.subject_ids.isin(clin_subs_existing), drop=True)

    #re-transform subsets into proper norm_data objects to get subclass methods
    clin_subset = NormData(data_vars=clin_subset.data_vars,coords=clin_subset.coords, attrs=clin_subset.attrs,name=clin_subset.name)
    remaining = NormData(data_vars=remaining.data_vars,coords=remaining.coords,attrs=remaining.attrs,name = remaining.name)

    #split and put clinical subjects back into the test set
    train, test = remaining.train_test_split(split)
    test = test.merge(clin_subset)
    return train, test

# Utility: grab trained models for each batch

In [ ]:
def get_models_ready_for_batch(new_batch_num, n_voxels, prev_batch_size, new_batch_size, model_dir):
    
    #voxel range
    start_vox = new_batch_num * new_batch_size
    end_vox = min((new_batch_num + 1) * new_batch_size, n_voxels)
    
    #map batch index to the corresponding prev batch index/indices to get list of previous batches to open
    prev_start_batch_idx = start_vox // prev_batch_size
    prev_end_batch_idx = (end_vox - 1) // prev_batch_size
    prev_batches = [f"batch_{x}" for x in range(prev_start_batch_idx, prev_end_batch_idx + 1)]#range deducts 1 so we add it back

    #load the first relevant model object from previous batch(es)
    model = NormativeModel.load(os.path.join(model_dir, prev_batches[0]))#stop at the batch directory, the toolkit will find the actual model file

    #collect regression models and variables
    response_vars = model.response_vars
    regression_models = model.regression_models
    outscalers = model.outscalers
 
    #if there's more relevant batched models, load them and collect regression models and variables
    if len(prev_batches)>1 :
        for prev_batch in prev_batches[1:]:
            prev_batch_model = NormativeModel.load(os.path.join(model_dir, prev_batch))
            response_vars.extend(prev_batch_model.response_vars)
            regression_models.update(prev_batch_model.regression_models)
            outscalers.update(prev_batch_model.outscalers)
            
    # subset everything to just the voxels that belong to this new batch
    vox_range = [f'voxel_{x}' for x in range(start_vox, end_vox)]
    response_vars = [vox for vox in response_vars if vox in vox_range] #the response vars were named accordingly during training
    regression_models = {vox: v for vox, v in regression_models.items() if vox in response_vars}
    outscalers = {vox: v for vox, v in outscalers.items() if vox in response_vars}
    
    #update the model object
    model.response_vars = response_vars
    model.regression_models = regression_models
    model.outscalers = outscalers

    return model

# Running transfer or extend with a torque/slurm cluster : 

In [ ]:
for b in range(new_n_batches)[:]: #slice this loop if not all batches at once
    batch = f'batch_{b}'
    print(f"Submitting job for {batch}")
    
    #grab transfer/extend data
    norm_data_path = os.path.join(out_dir, batch, 'norm_data.pkl')
    with open(norm_data_path, 'rb') as f:
        norm_data = dill.load(f)
    
    if clinical_set: #optional - put clinical subjects in test set
        train,test = put_patients_in_test_set(clin_subs, norm_data, split = split)
    else :#else just split
        train, test = norm_data.train_test_split(splits = split)
        
    #grab trained models 
    batch_model = get_models_ready_for_batch(
            new_batch_num = b, 
            n_voxels = n_voxels, 
            prev_batch_size = prev_batch_size, 
            new_batch_size = new_batch_size, 
            model_dir = model_dir, 
            )
    #setup runner
    runner = Runner(
        cross_validate=False,
        parallelize=True,
        n_batches = 2, #minimum is 2, here the Y data (voxel values) will be split (e.g. 2 jobs with 75 voxels each, for an initial batch_size of 150)
        #as the data have already been batched earlier due to RAM limitations, there should be no need for aggressive splitting here
        environment=venv_path,
        job_type="slurm",  # or "torque"
        time_limit="10:00:00", #as the pcntoolkit doesn't save all files before the end, best to use some padding here
        memory = "2GB", # adjust as needed, 2gb is comfortable for a job of 75 voxels
        n_cores=1, #some of the pcntoolkit computation is parallelized but for this use case (voxelwise BLR), multiple 1-core jobs should be faster overall than multicore
        log_dir=os.path.join(out_dir,'logs/'),
        temp_dir=os.path.join(out_dir,'tmp/'),
        preamble = "module load anaconda3" #whatever preamble is needed for the cluster job
    )
    
    #run transfer or extend - the runner submits jobs directly to the cluster, now we just have to monitor them
    if function == 'transfer' : 
        runner.transfer_predict(batch_model, train, test, save_dir = os.path.join(out_dir, batch), observe=False)
    
    if function == 'extend':
        runner.extend_predict(batch_model, train, test, save_dir = os.path.join(out_dir, batch), observe=False)

##syntax if we want to try one batch locally for testing purposes 
# batch_model.transfer_predict(train, test,save_dir = os.path.join(out_dir, batch))
# batch_model.extend_predict(train, test,save_dir = os.path.join(out_dir, batch))

# Running transfer or extend on a local machine

In [ ]:
#optionally slice the batch range - likely necessary for extend
first_batch = 0
last_batch = 10 # will actually be last_batch-1 because of range()

def run_transfer(b, model_dir, out_dir, n_voxels, prev_batch_size, new_batch_size, clinical_set, split):
    batch = f"batch_{b}"
    print(f'Transferring {batch}')
    log_file = os.path.join(out_dir, 'logs',f'transfer_log_{batch}.log')
    
    with open(log_file, "w", buffering = 1) as log_f, redirect_stdout(log_f), redirect_stderr(log_f):
        
        #grab transfer data and trained models
        norm_data_path = os.path.join(out_dir,batch, 'norm_data.pkl')
        with open(norm_data_path, 'rb') as f:
            norm_data = dill.load(f)
       
        if clinical_set: #optional - put clinical subjects in test set
            train,test = put_patients_in_test_set(clin_subs, norm_data, split = split)
        else :#else just split
            train, test = norm_data.train_test_split(splits = split)
        
        batch_model = get_models_ready_for_batch(
            new_batch_num = b, 
            n_voxels = n_voxels, 
            prev_batch_size = prev_batch_size, 
            new_batch_size = new_batch_size, 
            model_dir = model_dir, 
            )

        #run transfer
        batch_model.transfer_predict(train, test, save_dir = os.path.join(out_dir, batch))  

def run_extend(b, model_dir, out_dir, n_voxels, prev_batch_size, new_batch_size, clinical_set, split):
    batch = f"batch_{b}"
    print(f'Extending {batch}')
    log_file = os.path.join(out_dir, 'logs',f'transfer_log_{batch}.log')

    with open(log_file, "w", buffering = 1) as log_f, redirect_stdout(log_f), redirect_stderr(log_f):
        
        #grab extend data and trained models
        norm_data_path = os.path.join(out_dir,batch, 'norm_data.pkl')
        with open(norm_data_path, 'rb') as f:
            norm_data = dill.load(f)
       
        if clinical_set: #optional - put clinical subjects in test set
            train,test = put_patients_in_test_set(clin_subs, norm_data, split = split)
        else :#else just split
            train, test = norm_data.train_test_split(splits = split)
        
        batch_model = get_models_ready_for_batch(
            new_batch_num = b, 
            n_voxels = n_voxels, 
            prev_batch_size = prev_batch_size, 
            new_batch_size = new_batch_size, 
            model_dir = model_dir, 
            )
        
        #run extend
        batch_model.extend_predict(train, test,save_dir = os.path.join(out_dir, batch))

#if extend, try to run in chunks of batches - each 150 voxels batch is designed to run comfortably with one cpu core and 4GB of RAM
#Very rough estimates of runtime per core for a batch of 150 voxels : transfer = 8 mins, extend = 4-7 hours
if function == 'transfer': 
    Parallel(n_jobs=-1,verbose =10)(delayed(run_transfer)(b, model_dir, out_dir, n_voxels, prev_batch_size, new_batch_size, clinical_set, split) for b in range(new_n_batches)[first_batch: min(last_batch, new_n_batches)])
elif function == 'extend': 
    Parallel(n_jobs=-1,verbose =10)(delayed(run_extend)(b, model_dir, out_dir, n_voxels, prev_batch_size, new_batch_size, clinical_set, split) for b in range(new_n_batches)[first_batch: min(last_batch, new_n_batches)]) #adjust according to specs
else :
    raise ValueError(f"Unknown function: {function}")
    

# Get Z score brain maps of clinical subjects after transfer/extend

In [ ]:
if not os.path.exists(out_dir + '/visuals/'):
    os.makedirs(out_dir + '/visuals/')

for sub in clin_subs[:]:
    Z_full = []
    for b in range(new_n_batches)[:]: #grab Z data across all batches to build full nifti
        batch = f'batch_{b}'
        print(batch)
        
        with open(os.path.join(out_dir,batch, 'norm_data.pkl'), 'rb') as f:
            norm_data= pickle.load(f)

        #need to rebuild train and test sets here, because results are saved for these two and not for the general norm_data object
        if clinical_set: 
                train,test = put_patients_in_test_set(clin_subs, norm_data, split = split)
            else :
                train, test = norm_data.train_test_split(splits = split) 
                
        #now we have the proper object to load zscores
        test.load_zscores(save_dir = os.path.join(w_dir, batch, 'results'))
        
        idx = np.where(test.subject_ids.values == sub)[0]
        sub_batch = test.isel(observations=idx)

        Z_df = pd.DataFrame(sub_batch.Z.values, columns=sub_batch.Z.response_vars.values)
        Z_df.columns = [f'{batch}_{i}' for i in Z_df.columns]
        Z_full.append(Z_df)
    
    Z_subject = pd.concat(Z_full, axis = 1)
    nan_cols = Z_subject.columns[Z_subject.isna().any()].tolist() #check if there are nans if some batches or voxels failed
    print('Failed transfer/extend: ', nan_cols)
    ptksave(Z_subject, os.path.join(out_dir, 'visuals', f'{sub}.nii.gz'), example=ex_nii, mask=mask_nii)